# Analysing oscillator data with DuckDB

This notebook discovers the generated oscillator dataset, registers its Parquet files as DuckDB views, checks that the resulting relations have the expected grain, and provides model solutions for all five research questions in [`SCENARIO.md`](SCENARIO.md). The repository copy is intentionally output-free; execute it from top to bottom to produce local results.

Run this notebook with the repository's `oscillator` Pixi environment. If the ignored dataset is absent, generate it from the repository root with:

```bash
pixi run --environment oscillator generate-oscillator-data
```

## Data ingestion

### Setup

The path search supports kernels started from the repository root, `source-code`, or this notebook's directory. The manifest is the dataset contract: it records the version, layout, row counts, and generation parameters.

In [ ]:
import json
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt


def find_oscillator_directory() -> Path:
    candidates = (
        Path.cwd(),
        Path.cwd() / "oscillator",
        Path.cwd() / "source-code" / "oscillator",
    )
    for candidate in candidates:
        if (candidate / "generate_dataset.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate source-code/oscillator from the current directory."
    )


OSCILLATOR_DIR = find_oscillator_directory()
DATA_DIR = OSCILLATOR_DIR / "data"
MANIFEST_PATH = DATA_DIR / "manifest.json"

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "Dataset not found. From the repository root, run: "
        "pixi run --environment oscillator generate-oscillator-data"
    )

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
{
    "dataset_id": manifest["dataset_id"],
    "generator_version": manifest["generator_version"],
    **manifest["summary"],
}

### Register the Parquet datasets

The views keep the data in Parquet until a query requests rows. Hive partition discovery adds `damping_ratio` from directory names such as `damping_ratio=0.01`; DuckDB exposes this path-derived value as text. `filename` is retained so partition selection remains inspectable.

In [ ]:
connection = duckdb.connect()

runs_path = (DATA_DIR / "runs.parquet").as_posix()
solver_batches_path = (DATA_DIR / "solver_batches.parquet").as_posix()
observations_glob = (
    DATA_DIR / "observations" / "*" / "*.parquet"
).as_posix()

connection.read_parquet(runs_path).create_view("runs", replace=True)
connection.read_parquet(solver_batches_path).create_view(
    "solver_batches", replace=True
)
_ = connection.read_parquet(
    observations_glob,
    hive_partitioning=True,
    filename=True,
).create_view("observations", replace=True)

In [ ]:
connection.sql("DESCRIBE observations")

### Check the ingestion contract

Check row counts before interpreting the data. Each observation must join to exactly one run, every run must contain its declared number of samples, and the path-derived damping ratio must agree with the authoritative floating-point value in `runs.parquet`.

In [ ]:
connection.sql(
    """
    SELECT
        (SELECT count(*) FROM runs) AS runs,
        (SELECT count(*) FROM solver_batches) AS solver_batches,
        (SELECT count(*) FROM observations) AS observations,
        (
            SELECT count(*)
            FROM observations
            JOIN runs USING (run_id)
        ) AS joined_observations
    """
)

In [ ]:
connection.sql(
    """
    SELECT
        damping_ratio,
        count(*) AS observations,
        count(DISTINCT filename) AS files
    FROM observations
    GROUP BY damping_ratio
    ORDER BY try_cast(damping_ratio AS DOUBLE)
    """
)

In [ ]:
connection.sql(
    """
    WITH per_run AS (
        SELECT run_id, count(*) AS observed_samples
        FROM observations
        GROUP BY run_id
    ),
    run_quality AS (
        SELECT
            count(*) FILTER (
                WHERE observed_samples != sample_count
            ) AS sample_count_mismatches,
            count(*) FILTER (WHERE NOT completed) AS incomplete_runs
        FROM per_run
        JOIN runs USING (run_id)
    ),
    partition_quality AS (
        SELECT count(*) FILTER (
            WHERE try_cast(o.damping_ratio AS DOUBLE)
                  IS DISTINCT FROM r.damping_ratio
        ) AS partition_mismatches
        FROM observations AS o
        JOIN runs AS r USING (run_id)
    ),
    solver_quality AS (
        SELECT count(*) FILTER (WHERE NOT success) AS failed_batches
        FROM solver_batches
    )
    SELECT *
    FROM run_quality, partition_quality, solver_quality
    """
)

### Build an analysis-ready relation

Join observations to run metadata using `run_id`. Keep the path-derived partition value visible as `partition_damping_ratio`, but use `run_damping_ratio` for numerical analysis. The view is a query definition rather than a materialized copy of the million-row dataset.

In [ ]:
connection.sql(
    """
    CREATE OR REPLACE TEMP VIEW analysis_ready AS
    SELECT
        o.run_id,
        o.sample_index,
        o.drive_phase_rad,
        o.time_s,
        o.displacement_m,
        o.velocity_m_s,
        o.specific_energy_j_kg,
        o.damping_ratio AS partition_damping_ratio,
        r.damping_ratio AS run_damping_ratio,
        r.frequency_ratio,
        r.natural_angular_frequency_rad_s,
        r.drive_angular_frequency_rad_s,
        r.drive_acceleration_m_s2,
        r.samples_per_period,
        r.cycles
    FROM observations AS o
    JOIN runs AS r USING (run_id)
    """
)

connection.sql(
    """
    SELECT *
    FROM analysis_ready
    WHERE partition_damping_ratio = '0.01'
    ORDER BY run_id, sample_index
    LIMIT 10
    """
)

## Reference solutions: estimate response amplitude

The first research question asks what should count as response amplitude. The three subsections below provide complete reference implementations of the alternatives in `SCENARIO.md`. In practice, select the estimator that matches the scientific claim; use two or more only when their differences are themselves informative.

For a common comparison, all three solutions use the final five driving periods: periods 36 through 40, represented internally by zero-based period indices 35 through 39. The window is half-open, so the endpoint after period 40 is excluded. A late window reduces the early transient's influence, but it does **not** establish steady state; the second research question must compare successive windows.

Each subsection is intentionally self-contained, so executing every preview repeats some scans. For a much larger dataset, calculate the shared grouped statistics in one pass or materialize a compact window summary when comparing estimators.

In [ ]:
window_start_period = 35
window_periods = 5
window_stop_period = window_start_period + window_periods

connection.sql(
    f"""
    CREATE OR REPLACE TEMP VIEW measurement_window AS
    SELECT *
    FROM analysis_ready
    WHERE sample_index >= {window_start_period} * samples_per_period
      AND sample_index < {window_stop_period} * samples_per_period
    """
)

connection.sql(
    f"""
    WITH per_run AS (
        SELECT
            run_id,
            count(*) AS samples,
            max(samples_per_period) AS samples_per_period
        FROM measurement_window
        GROUP BY run_id
    ),
    phase_steps AS (
        SELECT
            drive_phase_rad
            - lag(drive_phase_rad) OVER (
                PARTITION BY run_id ORDER BY sample_index
            ) AS phase_step,
            samples_per_period
        FROM measurement_window
    )
    SELECT
        (SELECT min(samples) FROM per_run) AS minimum_samples,
        (SELECT max(samples) FROM per_run) AS maximum_samples,
        (
            SELECT count(*)
            FROM per_run
            WHERE samples != {window_periods} * samples_per_period
        ) AS incomplete_windows,
        (
            SELECT max(abs(
                phase_step - 2.0 * pi() / samples_per_period
            ))
            FROM phase_steps
            WHERE phase_step IS NOT NULL
        ) AS maximum_phase_step_error
    """
)

### Half peak-to-peak amplitude

For displacement samples $x_i$, estimate amplitude as $\hat{A}_{\mathrm{pp}} = (\max_i x_i - \min_i x_i)/2$. This measures total excursion, including any remaining transient, and can underestimate extrema that fall between stored samples.

In [ ]:
connection.sql(
    """
    CREATE OR REPLACE TEMP VIEW amplitude_peak_to_peak AS
    SELECT
        run_id,
        run_damping_ratio,
        frequency_ratio,
        'peak_to_peak' AS method,
        (max(displacement_m) - min(displacement_m)) / 2.0
            AS amplitude_m,
        NULL::DOUBLE AS residual_rms_m
    FROM measurement_window
    GROUP BY run_id, run_damping_ratio, frequency_ratio
    """
)

connection.sql(
    """
    SELECT *
    FROM amplitude_peak_to_peak
    ORDER BY run_damping_ratio, frequency_ratio
    LIMIT 10
    """
)

### Demeaned root-mean-square amplitude

For $n$ displacement samples with mean $\bar{x}$, estimate amplitude as $\hat{A}_{\mathrm{RMS}} = \sqrt{(2/n)\sum_i(x_i-\bar{x})^2}$. The aggregate form below is equivalent. It is exact for a single sinusoid sampled over complete periods, but treats all oscillatory components as response.

In [ ]:
connection.sql(
    """
    CREATE OR REPLACE TEMP VIEW amplitude_rms AS
    SELECT
        run_id,
        run_damping_ratio,
        frequency_ratio,
        'demeaned_rms' AS method,
        sqrt(greatest(
            0.0,
            2.0 * (
                avg(displacement_m * displacement_m)
                - power(avg(displacement_m), 2)
            )
        )) AS amplitude_m,
        NULL::DOUBLE AS residual_rms_m
    FROM measurement_window
    GROUP BY run_id, run_damping_ratio, frequency_ratio
    """
)

connection.sql(
    """
    SELECT *
    FROM amplitude_rms
    ORDER BY run_damping_ratio, frequency_ratio
    LIMIT 10
    """
)

### Harmonic least-squares amplitude

Project displacement onto the known driving phase $\phi_i$ using $x_i = b_0 + b_c\cos(\phi_i) + b_s\sin(\phi_i) + \epsilon_i$. Under the checked assumptions—uniform phase spacing, complete periods, a half-open window, complete samples, and equal weights—the least-squares coefficients reduce to grouped sums. The drive-frequency amplitude is $A = \sqrt{b_c^2+b_s^2}$; the residual RMS measures motion not explained by that harmonic.

In [ ]:
connection.sql(
    """
    CREATE OR REPLACE TEMP VIEW amplitude_harmonic AS
    WITH sufficient_statistics AS (
        SELECT
            run_id,
            run_damping_ratio,
            frequency_ratio,
            avg(displacement_m) AS b_0,
            2.0 * avg(
                displacement_m * cos(drive_phase_rad)
            ) AS b_c,
            2.0 * avg(
                displacement_m * sin(drive_phase_rad)
            ) AS b_s,
            avg(displacement_m * displacement_m) AS mean_x_squared
        FROM measurement_window
        GROUP BY run_id, run_damping_ratio, frequency_ratio
    )
    SELECT
        run_id,
        run_damping_ratio,
        frequency_ratio,
        'harmonic_least_squares' AS method,
        sqrt(b_c * b_c + b_s * b_s) AS amplitude_m,
        sqrt(greatest(
            0.0,
            mean_x_squared
            - b_0 * b_0
            - (b_c * b_c + b_s * b_s) / 2.0
        )) AS residual_rms_m
    FROM sufficient_statistics
    """
)

connection.sql(
    """
    SELECT *
    FROM amplitude_harmonic
    ORDER BY run_damping_ratio, frequency_ratio
    LIMIT 10
    """
)

### Compare the reference estimates

Combining the three compatible run-level relations makes differences visible without implying that every analysis should retain all three. The preview uses runs at the nominal resonance frequency ratio $r=1$.

In [ ]:
connection.sql(
    """
    CREATE OR REPLACE TEMP VIEW amplitude_estimates AS
    SELECT * FROM amplitude_peak_to_peak
    UNION ALL
    SELECT * FROM amplitude_rms
    UNION ALL
    SELECT * FROM amplitude_harmonic
    """
)

connection.sql(
    """
    SELECT *
    FROM amplitude_estimates
    WHERE abs(frequency_ratio - 1.0) < 1.0e-12
    ORDER BY run_damping_ratio, method
    """
)

## Model solution: assess whether runs have settled

The second research question requires evidence from successive windows, not merely a late amplitude estimate. This model solution uses the harmonic least-squares estimator because it provides both the drive-frequency amplitude and a residual measure. It divides every run into eight half-open windows of five driving periods, compares the final two, and retains every run in an explicit settling class.

### Estimate amplitude in successive windows

The terminal sample is excluded before assigning zero-based window identifiers. Each window therefore contains exactly 250 observations: five periods with 50 samples per period. The same grouped harmonic projection used above is then applied independently to every run and window. The compact window-level result is materialized as a temporary table because several later diagnostics reuse it; leaving it as a view would repeat the scan and aggregation.

In [ ]:
settling_window_periods = 5

connection.sql(
    f"""
    CREATE OR REPLACE TEMP TABLE harmonic_by_window AS
    WITH windowed AS (
        SELECT
            *,
            cast(floor(
                sample_index
                / (samples_per_period * {settling_window_periods}.0)
            ) AS INTEGER) AS window_id
        FROM analysis_ready
        WHERE sample_index < cycles * samples_per_period
    ),
    sufficient_statistics AS (
        SELECT
            run_id,
            run_damping_ratio,
            frequency_ratio,
            natural_angular_frequency_rad_s,
            drive_acceleration_m_s2,
            window_id,
            count(*) AS samples,
            avg(displacement_m) AS b_0,
            2.0 * avg(
                displacement_m * cos(drive_phase_rad)
            ) AS b_c,
            2.0 * avg(
                displacement_m * sin(drive_phase_rad)
            ) AS b_s,
            avg(displacement_m * displacement_m) AS mean_x_squared
        FROM windowed
        GROUP BY
            run_id,
            run_damping_ratio,
            frequency_ratio,
            natural_angular_frequency_rad_s,
            drive_acceleration_m_s2,
            window_id
    ),
    estimates AS (
        SELECT
            *,
            sqrt(b_c * b_c + b_s * b_s) AS amplitude_m,
            sqrt(greatest(
                0.0,
                mean_x_squared
                - b_0 * b_0
                - (b_c * b_c + b_s * b_s) / 2.0
            )) AS residual_rms_m
        FROM sufficient_statistics
    )
    SELECT
        run_id,
        run_damping_ratio,
        frequency_ratio,
        natural_angular_frequency_rad_s,
        drive_acceleration_m_s2,
        window_id,
        window_id * {settling_window_periods} + 1 AS first_period,
        (window_id + 1) * {settling_window_periods} AS last_period,
        samples,
        amplitude_m,
        residual_rms_m,
        residual_rms_m / nullif(amplitude_m, 0.0)
            AS residual_fraction
    FROM estimates
    """
)

connection.sql(
    """
    SELECT *
    FROM harmonic_by_window
    WHERE abs(run_damping_ratio - 0.01) < 1.0e-12
      AND abs(frequency_ratio - 1.0) < 1.0e-12
    ORDER BY window_id
    """
)

### Compare the final two windows

Rank windows from latest to earliest within each run, then compare the latest two. For amplitudes $A_1$ and $A_2$, the symmetric relative change is $\Delta_A = 2|A_2-A_1|/(A_1+A_2)$. Unlike division by only one amplitude, this definition treats the two windows symmetrically.

In [ ]:
connection.sql(
    """
    CREATE OR REPLACE TEMP TABLE late_window_comparison AS
    WITH ranked AS (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY run_id ORDER BY window_id DESC
            ) AS late_rank
        FROM harmonic_by_window
    )
    SELECT
        current.run_id,
        current.run_damping_ratio,
        current.frequency_ratio,
        current.natural_angular_frequency_rad_s,
        current.drive_acceleration_m_s2,
        previous.window_id AS previous_window_id,
        current.window_id AS final_window_id,
        previous.amplitude_m AS previous_amplitude_m,
        current.amplitude_m AS final_amplitude_m,
        2.0 * abs(current.amplitude_m - previous.amplitude_m)
            / nullif(
                current.amplitude_m + previous.amplitude_m, 0.0
            ) AS relative_amplitude_change,
        current.residual_rms_m AS final_residual_rms_m,
        current.residual_fraction AS final_residual_fraction
    FROM ranked AS current
    JOIN ranked AS previous
      ON current.run_id = previous.run_id
     AND current.late_rank = 1
     AND previous.late_rank = 2
    """
)

connection.sql(
    """
    SELECT *
    FROM late_window_comparison
    ORDER BY relative_amplitude_change DESC
    LIMIT 10
    """
)

### Classify without discarding runs

This model solution uses 1% as both the maximum acceptable amplitude change and the maximum residual-to-amplitude ratio. These are transparent demonstration criteria, not universal physical constants. A run is `still_changing` when its late amplitudes differ by more than 1%; `stable_amplitude_with_residual` when its amplitude is stable but more than 1% of it remains as residual RMS; and `stable_response` only when both criteria pass. The later sensitivity investigation should vary both thresholds.

In [ ]:
amplitude_change_tolerance = 0.01
residual_fraction_tolerance = 0.01
total_cycles = manifest["parameters"]["cycles"]
transient_envelope_target = 0.01

connection.sql(
    f"""
    CREATE OR REPLACE TEMP TABLE settling_assessment AS
    SELECT
        *,
        CASE
            WHEN relative_amplitude_change IS NULL
              OR final_residual_fraction IS NULL
                THEN 'insufficient_data'
            WHEN relative_amplitude_change
                 > {amplitude_change_tolerance}
                THEN 'still_changing'
            WHEN final_residual_fraction
                 > {residual_fraction_tolerance}
                THEN 'stable_amplitude_with_residual'
            ELSE 'stable_response'
        END AS settling_class,
        exp(
            -2.0 * pi() * run_damping_ratio * {total_cycles}
            / frequency_ratio
        ) AS expected_transient_factor,
        greatest(
            0,
            cast(ceil(
                -frequency_ratio * ln({transient_envelope_target})
                / (2.0 * pi() * run_damping_ratio)
            ) AS INTEGER) - {total_cycles}
        ) AS estimated_additional_periods
    FROM late_window_comparison
    """
)

connection.sql(
    """
    SELECT
        run_damping_ratio,
        settling_class,
        count(*) AS runs
    FROM settling_assessment
    GROUP BY run_damping_ratio, settling_class
    ORDER BY run_damping_ratio, settling_class
    """
)

### Compare with the transient-decay expectation

The homogeneous transient envelope is expected to decay by a factor $E_N = \exp(-2\pi\zeta N/r)$ after $N$ driving periods, where $\zeta$ is the damping ratio and $r$ is the driving-to-natural frequency ratio. Solving $E_N \leq \epsilon$ gives the planning estimate $N_\epsilon = \lceil-r\ln(\epsilon)/(2\pi\zeta)\rceil$. Here $\epsilon=0.01$. Neither $E_N$ nor $N_\epsilon$ is a measured residual or proof of convergence: the unknown initial transient magnitude and phase still matter. The summary checks whether the observed classifications follow the expected damping pattern; the row-level preview then identifies unresolved runs and gives a rough estimate of how many additional periods may be needed.

In [ ]:
connection.sql(
    """
    SELECT
        run_damping_ratio,
        min(expected_transient_factor)
            AS minimum_expected_transient_factor,
        max(expected_transient_factor)
            AS maximum_expected_transient_factor,
        max(relative_amplitude_change)
            AS maximum_observed_amplitude_change,
        count_if(settling_class != 'stable_response')
            AS unresolved_runs
    FROM settling_assessment
    GROUP BY run_damping_ratio
    ORDER BY run_damping_ratio
    """
)

Keep the unresolved runs in the result. Extending each simulation by at least the estimated number of periods, then repeating the adjacent-window test, is a defensible next step; it is not valid to reclassify a run from the estimate alone.

In [ ]:
connection.sql(
    """
    SELECT *
    FROM settling_assessment
    WHERE settling_class != 'stable_response'
    ORDER BY relative_amplitude_change DESC
    LIMIT 10
    """
)

## Model solution: quantify how damping changes resonance

This reference solution treats only `stable_response` runs as defensible for estimating a resonance peak or its width. That conservative choice avoids presenting a transient-contaminated peak as a steady-state result, but it can leave a damping ratio without a defensible peak. All runs remain in the analysis and visualization so that this loss of coverage is visible rather than silently filtered away.

### Calculate the dimensionless gain

For final-window harmonic amplitude $A$, natural angular frequency $\omega_0$, and driving-acceleration amplitude $a$, the dimensionless gain is $G=A\omega_0^2/a$. The compact run-level table retains the settling evidence beside $G$ so every subsequent claim can be traced back to its convergence assessment.

In [ ]:
connection.sql(
    """
    CREATE OR REPLACE TEMP TABLE resonance_response AS
    SELECT
        *,
        final_amplitude_m
            * power(natural_angular_frequency_rad_s, 2)
            / nullif(drive_acceleration_m_s2, 0.0) AS gain
    FROM settling_assessment
    """
)

connection.sql(
    """
    SELECT
        run_damping_ratio,
        settling_class,
        count(*) AS runs
    FROM resonance_response
    GROUP BY run_damping_ratio, settling_class
    ORDER BY run_damping_ratio, settling_class
    """
)

### Compare the resonance curves

Each panel uses the same frequency-ratio range but an independent gain scale, because a shared vertical scale would flatten the more strongly damped responses. The pale line connects sampled frequencies only; no smoothing or interpolation is used. Marker colour identifies whether each point supports a steady-state claim.

In [ ]:
response_for_plot = connection.sql(
    """
    SELECT
        run_damping_ratio,
        frequency_ratio,
        gain,
        settling_class
    FROM resonance_response
    ORDER BY run_damping_ratio, frequency_ratio
    """
).df()

status_styles = [
    ("stable_response", "tab:blue", "stable response"),
    (
        "stable_amplitude_with_residual",
        "tab:orange",
        "stable amplitude, residual remains",
    ),
    ("still_changing", "tab:red", "still changing"),
]
damping_ratios = sorted(response_for_plot["run_damping_ratio"].unique())
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True)

for ax, damping_ratio in zip(axes.flat, damping_ratios, strict=True):
    response = response_for_plot[
        response_for_plot["run_damping_ratio"] == damping_ratio
    ]
    ax.plot(
        response["frequency_ratio"],
        response["gain"],
        color="0.75",
        linewidth=1.0,
        zorder=1,
    )
    for settling_class, colour, label in status_styles:
        points = response[response["settling_class"] == settling_class]
        ax.scatter(
            points["frequency_ratio"],
            points["gain"],
            s=14,
            color=colour,
            label=label if ax is axes.flat[0] else None,
            zorder=2,
        )
    ax.set_title(f"damping ratio = {damping_ratio:g}")
    ax.grid(alpha=0.2)

fig.supxlabel("frequency ratio r", y=0.08)
fig.supylabel("dimensionless gain G")
fig.legend(
    *axes.flat[0].get_legend_handles_labels(),
    loc="lower center",
    bbox_to_anchor=(0.5, 0.0),
    ncol=3,
)
fig.tight_layout(rect=(0.0, 0.13, 1.0, 1.0))

### Locate each observed peak

The observed peak is the largest sampled gain for a damping ratio; ties are resolved toward the smaller frequency ratio. Its settling class, late-window amplitude change, and residual fraction are reported with the peak. `peak_defensible` is therefore an evidence flag, not another numerical threshold introduced after seeing the curve. The peak frequency is reported on the sampled grid and is not refined by interpolation.

In [ ]:
connection.sql(
    """
    CREATE OR REPLACE TEMP TABLE resonance_peaks AS
    WITH ranked AS (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY run_damping_ratio
                ORDER BY gain DESC, frequency_ratio
            ) AS peak_rank
        FROM resonance_response
    ),
    coverage AS (
        SELECT
            run_damping_ratio,
            count(*) AS total_runs,
            count_if(settling_class = 'stable_response')
                AS stable_runs
        FROM resonance_response
        GROUP BY run_damping_ratio
    )
    SELECT
        ranked.run_damping_ratio,
        coverage.total_runs,
        coverage.stable_runs,
        ranked.frequency_ratio AS observed_peak_frequency_ratio,
        ranked.gain AS observed_peak_gain,
        ranked.settling_class AS peak_settling_class,
        ranked.relative_amplitude_change
            AS peak_relative_amplitude_change,
        ranked.final_residual_fraction
            AS peak_residual_fraction,
        ranked.settling_class = 'stable_response'
            AS peak_defensible,
        ranked.gain / sqrt(2.0) AS half_power_gain
    FROM ranked
    JOIN coverage USING (run_damping_ratio)
    WHERE ranked.peak_rank = 1
    """
)

connection.sql(
    """
    SELECT * EXCLUDE (half_power_gain)
    FROM resonance_peaks
    ORDER BY run_damping_ratio
    """
)

### Estimate width and sharpness at grid resolution

For a defensible peak $G_\mathrm{max}$, the half-power threshold is $G_\mathrm{max}/\sqrt{2}$. Let $r_\mathrm{L,in}$ and $r_\mathrm{R,in}$ be the smallest and largest sampled frequency ratios at or above that threshold. If both crossings are bracketed by the grid, the half-power bandwidth $\Delta r$ lies between $r_\mathrm{R,in}-r_\mathrm{L,in}$ and that span plus two grid steps. The corresponding sharpness is reported as a quality-factor interval $Q=r_\mathrm{peak}/\Delta r$, where $r_\mathrm{peak}$ is the sampled peak frequency ratio. This interval assumes a single contiguous peak with monotonic threshold crossings; no sub-grid interpolation is claimed. A missing interval is a scientific result: either the peak is unsettled or the sampled frequency range does not bracket both crossings.

In [ ]:
frequency_grid_spacing = (
    manifest["parameters"]["frequency_max"]
    - manifest["parameters"]["frequency_min"]
) / (manifest["parameters"]["frequency_points"] - 1)

connection.sql(
    f"""
    CREATE OR REPLACE TEMP TABLE resonance_bandwidth AS
    WITH half_power_points AS (
        SELECT
            response.run_damping_ratio,
            min(response.frequency_ratio) AS lower_inside_ratio,
            max(response.frequency_ratio) AS upper_inside_ratio,
            count(*) AS half_power_samples
        FROM resonance_response AS response
        JOIN resonance_peaks AS peak USING (run_damping_ratio)
        WHERE peak.peak_defensible
          AND response.settling_class = 'stable_response'
          AND response.gain >= peak.half_power_gain
        GROUP BY response.run_damping_ratio
    ),
    grid_bounds AS (
        SELECT
            run_damping_ratio,
            min(frequency_ratio) AS grid_minimum_ratio,
            max(frequency_ratio) AS grid_maximum_ratio
        FROM resonance_response
        GROUP BY run_damping_ratio
    ),
    classified AS (
        SELECT
            peak.run_damping_ratio,
            peak.observed_peak_frequency_ratio,
            peak.peak_defensible,
            points.lower_inside_ratio,
            points.upper_inside_ratio,
            points.half_power_samples,
            points.upper_inside_ratio - points.lower_inside_ratio
                AS sampled_half_power_span,
            CASE
                WHEN NOT peak.peak_defensible
                    THEN 'peak_not_settled'
                WHEN points.lower_inside_ratio IS NULL
                    THEN 'no_stable_half_power_points'
                WHEN abs(
                    points.lower_inside_ratio
                    - grid.grid_minimum_ratio
                ) < 1.0e-12
                  OR abs(
                    points.upper_inside_ratio
                    - grid.grid_maximum_ratio
                ) < 1.0e-12
                    THEN 'not_bracketed_by_frequency_grid'
                ELSE 'bracketed_to_grid_resolution'
            END AS bandwidth_status
        FROM resonance_peaks AS peak
        JOIN grid_bounds AS grid USING (run_damping_ratio)
        LEFT JOIN half_power_points AS points
            USING (run_damping_ratio)
    ),
    reported AS (
        SELECT
            *,
            CASE WHEN bandwidth_status
                           = 'bracketed_to_grid_resolution'
                THEN sampled_half_power_span
            END AS half_power_width_lower,
            CASE WHEN bandwidth_status
                           = 'bracketed_to_grid_resolution'
                THEN sampled_half_power_span
                     + 2.0 * {frequency_grid_spacing}
            END AS half_power_width_upper
        FROM classified
    )
    SELECT
        run_damping_ratio,
        bandwidth_status,
        lower_inside_ratio,
        upper_inside_ratio,
        half_power_samples,
        half_power_width_lower,
        half_power_width_upper,
        observed_peak_frequency_ratio
            / nullif(half_power_width_upper, 0.0)
            AS quality_factor_lower,
        observed_peak_frequency_ratio
            / nullif(half_power_width_lower, 0.0)
            AS quality_factor_upper
    FROM reported
    """
)

connection.sql(
    """
    SELECT *
    FROM resonance_bandwidth
    ORDER BY run_damping_ratio
    """
)

The model solution supports the qualitative conclusion that increasing damping lowers and broadens the resonance response. It does **not** support a defensible peak or width for the least-damped ensemble because its observed peak has not settled. At the strongest damping, the peak is defensible but the lower half-power crossing lies outside the sampled frequency range, so the full bandwidth and $Q$ remain unresolved. These limitations should be carried into later comparisons rather than replaced by interpolated values.

## Model solution: compare with steady-state theory

The simulation and analytical expression describe the same idealized linear oscillator. Agreement is therefore a consistency check on the ODE integration, data reduction, grouping, and normalization—not independent experimental validation. Disagreement must first be examined for transient contamination and window dependence before it is attributed to the model or solver.

### Calculate the theoretical gain

For frequency ratio $r=\omega_\mathrm{d}/\omega_0$ and damping ratio $\zeta$, the steady-state dimensionless gain is $G_\mathrm{theory}=[(1-r^2)^2+(2\zeta r)^2]^{-1/2}$, where $\omega_\mathrm{d}$ is the driving angular frequency and $\omega_0$ is the natural angular frequency. Define the signed difference as $d=G_\mathrm{estimated}-G_\mathrm{theory}$ and the relative difference as $\delta=d/G_\mathrm{theory}$. The sampled parameter grid has $\zeta>0$, so $G_\mathrm{theory}$ is finite and positive; the diagnostic below verifies that its denominator is safely away from zero before relative differences are interpreted.

In [ ]:
connection.sql(
    """
    CREATE OR REPLACE TEMP TABLE theory_comparison AS
    WITH predicted AS (
        SELECT
            *,
            1.0 / sqrt(
                power(1.0 - power(frequency_ratio, 2), 2)
                + power(
                    2.0 * run_damping_ratio * frequency_ratio,
                    2
                )
            ) AS theoretical_gain
        FROM resonance_response
    )
    SELECT
        *,
        gain - theoretical_gain AS signed_gain_difference,
        (gain - theoretical_gain)
            / nullif(theoretical_gain, 0.0)
            AS relative_gain_difference
    FROM predicted
    """
)

connection.sql(
    """
    SELECT
        min(theoretical_gain) AS minimum_theoretical_gain,
        max(theoretical_gain) AS maximum_theoretical_gain,
        count_if(relative_gain_difference IS NULL)
            AS undefined_relative_differences
    FROM theory_comparison
    """
)

### Compare the final-window estimates

Summarize discrepancies separately by damping ratio and settling class. The median absolute relative difference describes a typical discrepancy, while the maximum exposes localized failures that a median could hide. The second table deliberately focuses on the resonance neighbourhood $|r-1|\leq0.2$, where both the gain and sensitivity to incomplete settling are largest.

In [ ]:
connection.sql(
        """
        SELECT
            run_damping_ratio,
            settling_class,
            count(*) AS runs,
            avg(signed_gain_difference)
                AS mean_signed_gain_difference,
            median(abs(relative_gain_difference))
                AS median_absolute_relative_difference,
            max(abs(relative_gain_difference))
                AS maximum_absolute_relative_difference
        FROM theory_comparison
        GROUP BY run_damping_ratio, settling_class
        ORDER BY run_damping_ratio, settling_class
        """
).show(max_rows=30)

connection.sql(
    """
    SELECT
        run_damping_ratio,
        frequency_ratio,
        gain AS estimated_gain,
        theoretical_gain,
        signed_gain_difference,
        relative_gain_difference,
        settling_class
    FROM theory_comparison
    WHERE abs(frequency_ratio - 1.0) <= 0.2
    ORDER BY abs(relative_gain_difference) DESC
    LIMIT 12
    """
).show(max_rows=30)

### Locate discrepancies across frequency

The panels show $100\delta$, the signed relative difference in percent, with zero denoting agreement. Independent vertical scales keep small discrepancies visible for the well-settled ensembles; they therefore must not be used to compare apparent vertical magnitudes between panels. Points remain classified by settling evidence, and no smoothing is applied.

In [ ]:
error_for_plot = connection.sql(
    """
    SELECT
        run_damping_ratio,
        frequency_ratio,
        100.0 * relative_gain_difference
            AS relative_gain_difference_percent,
        settling_class
    FROM theory_comparison
    ORDER BY run_damping_ratio, frequency_ratio
    """
).df()

fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True)
for ax, damping_ratio in zip(axes.flat, damping_ratios, strict=True):
    errors = error_for_plot[
        error_for_plot["run_damping_ratio"] == damping_ratio
    ]
    ax.axhline(0.0, color="0.5", linewidth=0.8)
    ax.plot(
        errors["frequency_ratio"],
        errors["relative_gain_difference_percent"],
        color="0.75",
        linewidth=1.0,
        zorder=1,
    )
    for settling_class, colour, label in status_styles:
        points = errors[errors["settling_class"] == settling_class]
        ax.scatter(
            points["frequency_ratio"],
            points["relative_gain_difference_percent"],
            s=14,
            color=colour,
            label=label if ax is axes.flat[0] else None,
            zorder=2,
        )
    ax.set_title(f"damping ratio = {damping_ratio:g}")
    ax.grid(alpha=0.2)

fig.supxlabel("frequency ratio r", y=0.08)
fig.supylabel("signed relative gain difference (%)")
fig.legend(
    *axes.flat[0].get_legend_handles_labels(),
    loc="lower center",
    bbox_to_anchor=(0.5, 0.0),
    ncol=3,
)
fig.tight_layout(rect=(0.0, 0.13, 1.0, 1.0))

### Test the comparison across analysis windows

A final-window comparison alone cannot distinguish persistent model disagreement from a transient that is still decaying. Recalculate the gain difference for every five-period window, then summarize the median and worst absolute relative difference across the sampled frequency ratios. The plot uses the maximum deliberately: it asks whether any part of the resonance curve remains poorly represented, rather than allowing well-behaved off-resonance runs to dominate an average.

In [ ]:
connection.sql(
    """
    CREATE OR REPLACE TEMP TABLE window_theory_comparison AS
    WITH gains AS (
        SELECT
            *,
            amplitude_m
                * power(natural_angular_frequency_rad_s, 2)
                / nullif(drive_acceleration_m_s2, 0.0)
                AS estimated_gain,
            1.0 / sqrt(
                power(1.0 - power(frequency_ratio, 2), 2)
                + power(
                    2.0 * run_damping_ratio * frequency_ratio,
                    2
                )
            ) AS theoretical_gain
        FROM harmonic_by_window
    )
    SELECT
        *,
        estimated_gain - theoretical_gain
            AS signed_gain_difference,
        (estimated_gain - theoretical_gain)
            / nullif(theoretical_gain, 0.0)
            AS relative_gain_difference
    FROM gains
    """
)

connection.sql(
    """
    CREATE OR REPLACE TEMP TABLE window_error_summary AS
    SELECT
        run_damping_ratio,
        window_id,
        first_period,
        last_period,
        median(abs(relative_gain_difference))
            AS median_absolute_relative_difference,
        max(abs(relative_gain_difference))
            AS maximum_absolute_relative_difference
    FROM window_theory_comparison
    GROUP BY
        run_damping_ratio, window_id, first_period, last_period
    """
)

connection.sql(
        """
        SELECT *
        FROM window_error_summary
        WHERE window_id IN (0, 7)
        ORDER BY run_damping_ratio, window_id
        """
).show(max_rows=30)

window_errors_for_plot = connection.sql(
    """
    SELECT *
    FROM window_error_summary
    ORDER BY run_damping_ratio, window_id
    """
).df()

fig, ax = plt.subplots(figsize=(8, 5))
for damping_ratio in damping_ratios:
    errors = window_errors_for_plot[
        window_errors_for_plot["run_damping_ratio"]
        == damping_ratio
    ]
    ax.plot(
        errors["last_period"],
        100.0 * errors["maximum_absolute_relative_difference"],
        marker="o",
        label=f"damping ratio = {damping_ratio:g}",
    )
ax.set_yscale("log")
ax.set_xlabel("last period in five-period window")
ax.set_ylabel("maximum absolute relative difference (%)")
ax.grid(alpha=0.2, which="both")
ax.legend(ncol=2)
fig.tight_layout()

For $\zeta\geq0.03$, every run is classified as stable and the largest final-window relative discrepancy falls from about 0.13% at $\zeta=0.03$ to the numerical-noise scale for the more strongly damped ensembles. The $\zeta=0.01$ result is more instructive: the worst discrepancy is about 9.5% near resonance, and even the provisional `stable_response` subgroup contains discrepancies of about 6.9%. Two adjacent windows can therefore appear stable while a slowly decaying oscillatory transient remains. The discrepancies shrink across successive windows, so the pattern supports transient contamination rather than a failure of the steady-state expression. Agreement still does not independently validate the physical model because both calculations encode the same linear oscillator assumptions; instead, this comparison exposes a limitation of the current settling rule that should be tested in the sensitivity analysis.

## Model solution: test sensitivity to analytical choices

A sensitivity analysis should target choices capable of changing the scientific conclusion. This solution first varies the settling criteria, which determine which runs support a steady-state claim, and then compares the three amplitude estimators, which summarize different aspects of the motion. The objective is not to select whichever result is most convenient, but to identify conclusions shared by credible alternatives and expose those that remain conditional.

### Vary the settling tolerances

Let $\tau_A$ be the maximum accepted relative amplitude change between the final two windows and $\tau_R$ the maximum accepted residual-to-amplitude ratio. Compare paired strict, baseline, and permissive criteria with $\tau_A=\tau_R$ equal to 0.5%, 1%, and 2%, respectively. Varying both together is deliberately compact, but its disadvantage is that a changed classification cannot be attributed to one threshold. The peak assessment asks whether the largest observed gain becomes defensible under any credible scenario.

In [ ]:
connection.sql(
    """
    CREATE OR REPLACE TEMP TABLE tolerance_scenarios AS
    SELECT *
    FROM (VALUES
        (1, 'strict', 0.005, 0.005),
        (2, 'baseline', 0.010, 0.010),
        (3, 'permissive', 0.020, 0.020)
    ) AS scenarios(
        scenario_order,
        scenario,
        amplitude_change_tolerance,
        residual_fraction_tolerance
    )
    """
)

connection.sql(
    """
    CREATE OR REPLACE TEMP TABLE tolerance_sensitivity AS
    SELECT
        scenarios.*,
        comparison.run_id,
        comparison.run_damping_ratio,
        comparison.frequency_ratio,
        CASE
            WHEN comparison.relative_amplitude_change IS NULL
              OR comparison.final_residual_fraction IS NULL
                THEN 'insufficient_data'
            WHEN comparison.relative_amplitude_change
                 > scenarios.amplitude_change_tolerance
                THEN 'still_changing'
            WHEN comparison.final_residual_fraction
                 > scenarios.residual_fraction_tolerance
                THEN 'stable_amplitude_with_residual'
            ELSE 'stable_response'
        END AS sensitivity_settling_class
    FROM late_window_comparison AS comparison
    CROSS JOIN tolerance_scenarios AS scenarios
    """
)

connection.sql(
        """
        SELECT
            scenario,
            amplitude_change_tolerance,
            residual_fraction_tolerance,
            run_damping_ratio,
            count_if(
                sensitivity_settling_class = 'stable_response'
            ) AS stable_response_runs,
            count_if(
                sensitivity_settling_class
                    = 'stable_amplitude_with_residual'
            ) AS residual_runs,
            count_if(
                sensitivity_settling_class = 'still_changing'
            ) AS changing_runs
        FROM tolerance_sensitivity
        GROUP BY
            scenario_order,
            scenario,
            amplitude_change_tolerance,
            residual_fraction_tolerance,
            run_damping_ratio
        ORDER BY scenario_order, run_damping_ratio
        """
).show(max_rows=30)

connection.sql(
    """
    WITH ranked_peaks AS (
        SELECT
            run_id,
            run_damping_ratio,
            frequency_ratio,
            gain,
            row_number() OVER (
                PARTITION BY run_damping_ratio
                ORDER BY gain DESC, frequency_ratio
            ) AS peak_rank
        FROM resonance_response
    )
    SELECT
        sensitivity.scenario,
        peaks.run_damping_ratio,
        peaks.frequency_ratio AS observed_peak_frequency_ratio,
        peaks.gain AS observed_peak_gain,
        sensitivity.sensitivity_settling_class
            AS peak_settling_class,
        sensitivity.sensitivity_settling_class = 'stable_response'
            AS peak_defensible
    FROM ranked_peaks AS peaks
    JOIN tolerance_sensitivity AS sensitivity USING (run_id)
    WHERE peaks.peak_rank = 1
    ORDER BY sensitivity.scenario_order, peaks.run_damping_ratio
    """
).show(max_rows=30)

### Compare the amplitude estimators

Compare half peak-to-peak amplitude, demeaned RMS amplitude, and harmonic least-squares amplitude over the same final five-period window. For estimator $m$, define $\Delta_m=(A_m-A_\mathrm{H})/A_\mathrm{H}$, where $A_m$ is its amplitude and $A_\mathrm{H}$ is the harmonic estimate for the same run. The harmonic estimate is the reference because it isolates the drive-frequency component; the alternatives include transient or other residual motion and are therefore diagnostic rather than automatically interchangeable. Report both observed peaks and estimator differences, retaining the baseline settling class.

In [ ]:
connection.sql(
    """
    CREATE OR REPLACE TEMP TABLE estimator_sensitivity AS
    SELECT
        estimates.run_id,
        estimates.run_damping_ratio,
        estimates.frequency_ratio,
        estimates.method,
        estimates.amplitude_m,
        estimates.amplitude_m
            * power(assessment.natural_angular_frequency_rad_s, 2)
            / nullif(assessment.drive_acceleration_m_s2, 0.0)
            AS gain,
        assessment.settling_class
    FROM amplitude_estimates AS estimates
    JOIN settling_assessment AS assessment USING (run_id)
    """
)

connection.sql(
    """
    CREATE OR REPLACE TEMP TABLE estimator_peaks AS
    WITH ranked AS (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY method, run_damping_ratio
                ORDER BY gain DESC, frequency_ratio
            ) AS peak_rank
        FROM estimator_sensitivity
    )
    SELECT
        method,
        run_damping_ratio,
        frequency_ratio AS observed_peak_frequency_ratio,
        gain AS observed_peak_gain,
        settling_class AS peak_settling_class,
        settling_class = 'stable_response' AS peak_defensible
    FROM ranked
    WHERE peak_rank = 1
    """
)

connection.sql(
        """
        SELECT *
        FROM estimator_peaks
        ORDER BY run_damping_ratio, method
        """
).show(max_rows=30)

connection.sql(
    """
    WITH harmonic_reference AS (
        SELECT
            run_id,
            amplitude_m AS harmonic_amplitude_m
        FROM estimator_sensitivity
        WHERE method = 'harmonic_least_squares'
    )
    SELECT
        estimates.run_damping_ratio,
        estimates.method,
        count(*) AS baseline_stable_runs,
        median(abs(
            (estimates.amplitude_m - reference.harmonic_amplitude_m)
            / nullif(reference.harmonic_amplitude_m, 0.0)
        )) AS median_absolute_relative_difference,
        max(abs(
            (estimates.amplitude_m - reference.harmonic_amplitude_m)
            / nullif(reference.harmonic_amplitude_m, 0.0)
        )) AS maximum_absolute_relative_difference
    FROM estimator_sensitivity AS estimates
    JOIN harmonic_reference AS reference USING (run_id)
    WHERE estimates.settling_class = 'stable_response'
    GROUP BY estimates.run_damping_ratio, estimates.method
    ORDER BY estimates.run_damping_ratio, estimates.method
    """
).show(max_rows=30)

Some conclusions are robust across these analytical choices. Increasing the damping ratio $\zeta$ lowers and broadens the resonance, and the observed peak for $\zeta = 0.01$ remains classified as `still_changing` under every tolerance scenario and with every amplitude estimator. There is therefore no defensible range of peak values to report for that ensemble: the result is unresolved. For $\zeta \geq 0.03$, the observed peak remains defensible under all three tolerance scenarios.

Other details are choice-dependent. For $\zeta = 0.01$, the number of runs labelled `stable_response` is 0, 8, or 18 under the strict, baseline, and permissive scenarios, respectively; the strict scenario also reclassifies 8 runs at $\zeta = 0.03$ as `stable_amplitude_with_residual`. Because both tolerances were changed together, this comparison cannot identify which criterion caused each reclassification.

Estimator choice changes little at the sampled frequencies, but not literally nothing. At $\zeta = 0.4$, peak-to-peak selects frequency ratio $r = 0.83$, whereas demeaned RMS and harmonic least-squares select $r = 0.82$. The difference $\Delta r = 0.01$ is exactly one frequency-grid step, so the data do not support a more precise peak location. Finally, the close estimator agreement at $\zeta = 0.01$ is **not** evidence that the response has settled: all three methods can summarize the same transient-contaminated record.

## Conclude the investigation

Use the expected-evidence checklist in [`SCENARIO.md`](SCENARIO.md) to assemble the final argument. Separate supported resonance trends from the unresolved lightly damped runs, state the sampled frequency resolution, and report which conclusions survived the sensitivity analysis.